# Sequence Models : **Recurrent Neural Network**

Dataset

      ↓
Preprocessing

      ↓
Tokenization

      ↓
Vocabulary

      ↓
Word IDs

      ↓
Padding

      ↓
Tensor Conversion

      ↓
DataLoader

      ↓
Embedding Layer

      ↓
RNN

      ↓
Linear Layer

      ↓
Prediction

      ↓
Loss

      ↓
Backpropagation Through Time

      ↓
Optimizer

      ↓
Evaluation

      ↓
Custom Prediction
**bold text**

# **Import Libraries**

In [27]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

# **Loading the Dataset**

In [28]:
!wget https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip
!unzip -o smsspamcollection.zip

--2026-07-29 16:29:11--  https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip
Resolving archive.ics.uci.edu (archive.ics.uci.edu)... 128.195.10.252
Connecting to archive.ics.uci.edu (archive.ics.uci.edu)|128.195.10.252|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified
Saving to: ‘smsspamcollection.zip.1’

smsspamcollection.z     [  <=>               ] 198.65K   512KB/s    in 0.4s    

2026-07-29 16:29:12 (512 KB/s) - ‘smsspamcollection.zip.1’ saved [203415]

Archive:  smsspamcollection.zip
  inflating: SMSSpamCollection       
  inflating: readme                  


In [29]:
import pandas as pd

df = pd.read_csv(
    "SMSSpamCollection",
    sep="\t",
    header=None,
    names=["label", "text"]
)

df.head()

,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [30]:
print(df.shape)

df["label"].value_counts()

(5572, 2)


,count
label,
ham,4825
spam,747


In [31]:
df["label"] = df["label"].map({
    "ham":0,
    "spam":1
}).astype(int)

print(df.head())

print(df.dtypes)

   label                                               text
0      0  Go until jurong point, crazy.. Available only ...
1      0                      Ok lar... Joking wif u oni...
2      1  Free entry in 2 a wkly comp to win FA Cup fina...
3      0  U dun say so early hor... U c already then say...
4      0  Nah I don't think he goes to usf, he lives aro...
label     int64
text     object
dtype: object


In [32]:
x = df["text"]
y = df["label"]

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42
)

# **Tokenization**

In [33]:
# Convert all text to lowercase.
x_train = x_train.str.lower()
x_test = x_test.str.lower()

#Tokenize every sentence.
x_train_tokens = x_train.str.split()

x_test_tokens = x_test.str.split()

# **Vocabulary Building & Padding**

In [34]:
from collections import Counter

word_counter = Counter()

for sentence in x_train_tokens:
    word_counter.update(sentence)

# create the vocabulary
vocab = {
    "<PAD>": 0,
    "<UNK>": 1
}

for word in word_counter:
    vocab[word] = len(vocab)
print("Vocabulary Size:", len(vocab))
list(vocab.items())[:20]

Vocabulary Size: 11811


[('<PAD>', 0),
 ('<UNK>', 1),
 ('reply', 2),
 ('to', 3),
 ('win', 4),
 ('£100', 5),
 ('weekly!', 6),
 ('where', 7),
 ('will', 8),
 ('the', 9),
 ('2006', 10),
 ('fifa', 11),
 ('world', 12),
 ('cup', 13),
 ('be', 14),
 ('held?', 15),
 ('send', 16),
 ('stop', 17),
 ('87239', 18),
 ('end', 19)]

In [35]:
# Create a function to convert words into IDs.
def text_to_ids(tokens, vocab):
    ids = []

    for word in tokens:
        ids.append(vocab.get(word, vocab["<UNK>"]))

    return ids

# Convert the training data
x_train_ids = x_train_tokens.apply(
    lambda sentence: text_to_ids(sentence, vocab)
)

# Convert the testing data.
x_test_ids = x_test_tokens.apply(
    lambda sentence: text_to_ids(sentence, vocab)
)

print(x_train_tokens.iloc[0])

print(x_train_ids.iloc[0])

['reply', 'to', 'win', '£100', 'weekly!', 'where', 'will', 'the', '2006', 'fifa', 'world', 'cup', 'be', 'held?', 'send', 'stop', 'to', '87239', 'to', 'end', 'service']
[2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 3, 18, 3, 19, 20]


# **Padding Sequence**

In [36]:
# -------------------------------
# Padding Sequences
# -------------------------------

# Find the maximum sequence length from the training data
max_length = max(len(sentence) for sentence in x_train_ids)

print("Maximum Sequence Length:", max_length)


# Function to pad a sequence with <PAD> (0)
def pad_sequence(sequence, max_length):
    return sequence + [0] * (max_length - len(sequence))


# Pad the training data
x_train_padded = x_train_ids.apply(
    lambda sentence: pad_sequence(sentence, max_length)
)

# Pad the testing data using the SAME max_length
x_test_padded = x_test_ids.apply(
    lambda sentence: pad_sequence(sentence, max_length)
)


# Verify that all sequences have the same length
print(len(x_train_padded.iloc[0]))
print(len(x_train_padded.iloc[1]))
print(len(x_train_padded.iloc[2]))


# Display one padded sequence
print("\nExample Padded Sequence:")
print(x_train_padded.iloc[0])

Maximum Sequence Length: 171
171
171
171

Example Padded Sequence:
[2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 3, 18, 3, 19, 20, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


# **Convert Input to Tensor**

In [37]:
x_train_tensor = torch.tensor(
    x_train_padded.tolist(),
    dtype=torch.long
)

x_test_tensor = torch.tensor(
    x_test_padded.tolist(),
    dtype=torch.long
)

y_train_tensor = torch.tensor(
    y_train.values,
    dtype=torch.float32
).view(-1,1)

y_test_tensor = torch.tensor(
    y_test.values,
    dtype=torch.float32
).view(-1,1)

print(x_train_tensor.shape)

print(y_train_tensor.shape)

torch.Size([4457, 171])
torch.Size([4457, 1])


# **TensorDataset**

In [38]:
train_dataset = TensorDataset(
    x_train_tensor,
    y_train_tensor
)

test_dataset = TensorDataset(
    x_test_tensor,
    y_test_tensor
)

# DataLoader
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

# **Build the Model**

In [39]:
class SpamRNN(nn.Module):

    def __init__(self):

        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=len(vocab),
            embedding_dim=100,
            padding_idx=0
        )

        self.rnn = nn.RNN(
            input_size=100,
            hidden_size=128,
            batch_first=True
        )

        self.fc = nn.Linear(
            128,
            1
        )

    def forward(self, x):

        x = self.embedding(x)

        output, hidden = self.rnn(x)

        hidden = hidden.squeeze(0)

        output = self.fc(hidden)

        return output


model = SpamRNN()

print(model)

SpamRNN(
  (embedding): Embedding(11811, 100, padding_idx=0)
  (rnn): RNN(100, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=1, bias=True)
)


# **Training Loop**

In [40]:
import torch.optim as optim

criterion = nn.BCEWithLogitsLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

num_epochs = 10

for epoch in range(num_epochs):

    model.train()

    total_loss = 0

    for X_batch, y_batch in train_loader:

        predictions = model(X_batch)

        loss = criterion(
            predictions,
            y_batch
        )

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch+1}/{num_epochs}  Loss: {total_loss/len(train_loader):.4f}"
    )

Epoch 1/10  Loss: 0.4115
Epoch 2/10  Loss: 0.3957
Epoch 3/10  Loss: 0.3960
Epoch 4/10  Loss: 0.3985
Epoch 5/10  Loss: 0.3955
Epoch 6/10  Loss: 0.3951
Epoch 7/10  Loss: 0.3948
Epoch 8/10  Loss: 0.3944
Epoch 9/10  Loss: 0.3957
Epoch 10/10  Loss: 0.3967


# **Evaluation**

In [41]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        outputs = model(X_batch)

        predictions = (torch.sigmoid(outputs) >= 0.5).float()

        correct += (predictions == y_batch).sum().item()

        total += y_batch.size(0)

accuracy = correct / total

print(f"Test Accuracy: {accuracy*100:.2f}%")

Test Accuracy: 86.64%


# **Testing**

In [45]:
model.eval()

while True:

    message = input("\nEnter your SMS (type 'exit' to quit): ")

    if message.lower() == "exit":
        break

    # Preprocess the input
    message = message.lower().split()

    # Convert words to IDs
    message = text_to_ids(message, vocab)

    # Pad the sequence
    message = pad_sequence(message, max_length)

    # Convert to tensor
    message = torch.tensor([message], dtype=torch.long)

    # Predict
    with torch.no_grad():
        output = model(message)
        probability = torch.sigmoid(output).item()

    print(f"Spam Probability: {probability:.4f}")

    if probability >= 0.5:
        print("Prediction: 🚨 Spam")
    else:
        print("Prediction: ✅ Ham")


Enter your SMS (type 'exit' to quit): exit
